# Measurement Model Testing (mirt)
**Finding the Optimal Latent Structure of Neuroticism Using Item Response Theory (IRT)**

This notebook contains the computational pipeline and empirical results for the measurement models evaluated. The analyses presented here utilize IRT to identify the most robust latent representation of the neuroticism construct.

This notebook displays the **real analytical results derived from the official HCP-YA 2025 data release**. While the restricted-access HCP dataset was used as the primary input for these calculations, the notebook has been carefully curated to ensure that **no individual-level data** are displayed. All outputs - including factor loadings, fit indices, and item parameters - represent aggregate statistical estimates across the full sample (*N*>1000). Consequently, these results provide a comprehensive view of the model performance without compromising the privacy of the participants or violating the HCP Restricted Data Use Agreement.

The structure of this notebook is as follows: <br><br>
**Contents:**
- Initialization:
    - Imports
    - Loading data
- Fitting the models: 
    - Unidimensional GRM unconstrained
    - Bifactor (anxiety & depression)
    - Bifactor (anxiety & depression, 2 items respectively)
    - Bifactor (anxiety, depression, self-consciousness, vulnerability)
- Comparing models:
    - model summaries 
    - M2 statistics
    - anova
- Model Selection Rationale

---
### Initialization

In [ ]:
# Set-Up 
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)
from rpy2 import robjects as ro
from rpy2.robjects import pandas2ri, numpy2ri
from rpy2.robjects.packages import importr
pandas2ri.activate()
numpy2ri.activate()
ro.r('set.seed(123)')
%load_ext rpy2.ipython
importr('base')
importr('MPsychoR')
importr('Gifi')
importr('psych')
importr('stats')
importr('eRm')
importr('ltm')
importr('mirt')

# Load prepped data
# The notebook was executed with official HCP data before the file path was changed. 
# Please update the path below to your local version of the prepped official data.
neuroticism_df = pd.read_csv(r"YOUR\PATH\HERE\prepped_RESTRICTED_data_for_SEM.csv", dtype={'Subject': str})
neuro_items = neuroticism_df.drop(columns=['Subject'])
print(neuro_items.columns)
# Put data into R
ro.globalenv['neuro_items'] = neuro_items

Index(['NEORAW_01', 'NEORAW_06', 'NEORAW_11', 'NEORAW_16', 'NEORAW_21',
       'NEORAW_26', 'NEORAW_31', 'NEORAW_36', 'NEORAW_41', 'NEORAW_46',
       'NEORAW_51', 'NEORAW_56'],
      dtype='object')


---
### Fitting the Models

Unidimensional Graded Response Model (unconstrained)

In [ ]:
%%R
grm_spec <- '
g = 1-12            # general factor neuroticism loading on all items
'         

grm_fit <- mirt(data = neuro_items, model = grm_spec, itemtype = "graded", SE = TRUE, verbose = FALSE)

Bifactor Model - Included Facets: Anxiety and Depression (all items)

In [3]:
%%R
bi_anx_dep_all_spec <- '
  g = 1-12            # general factor neuroticism loading on all items
  s_anx = 1, 5, 7     # anxiety facet (NEORAW_01, _21, _31)
  s_dep = 4, 6, 9, 10 # depression facet (NEORAW_16, _26, _41, _46)

  COV = g*s_anx(0), g*s_dep(0), s_anx*s_dep(0) # all factore uncorrelated
'

bi_anx_dep_all_fit <- mirt(data = neuro_items, model = bi_anx_dep_all_spec, itemtype = "graded", SE = TRUE, verbose = FALSE)

Bifactor Model - Facets Included: Anxiety, Depression, Self-Consciousness, and Vulnerability (all facets comprising at least two items with all items, respectively)

In [4]:
%%R
bi_4fac_all_spec <- '
  g = 1-12            # general factor neuroticism loading on all items
  s_anx = 1, 5, 7     # anxiety facet (NEORAW_01, _21, _31)
  s_dep = 4, 6, 9, 10 # depression facet (NEORAW_16, _26, _41, _46)
  s_selfcon = 2, 12   # self-consciousness (NEORAW_06, _56)
  s_vul = 3, 11       # vulnerability (NEORAW_11, _51)

  COV = g*s_anx(0), g*s_dep(0), g*s_selfcon(0), g*s_vul(0), # all factore uncorrelated
        s_anx*s_dep(0), s_anx*s_selfcon(0), s_anx*s_vul(0),
        s_dep*s_selfcon(0), s_dep*s_vul(0),
        s_selfcon*s_vul(0)

'

bi_4fac_all_fit <- mirt(data = neuro_items, model = bi_4fac_all_spec, itemtype = "graded", SE = TRUE, verbose = FALSE)

R[write to console]: EM quadrature for high dimensional models are better handled
                                 	with the "QMCEM" or "MCEM" method



🌟 Bifactor Model - Included Facets: Anxiety and Depression (2 strongest-loading items each)

In [ ]:
%%R
bi_anx_dep_2_spec <- '
  g     = 1-12        # general factor neuroticism loading on all items
  s_anx = 1, 7        # anxiety facet (NEORAW_01, _31)
  s_dep = 4, 10       # depression facet (NEORAW_16, _46)

  COV = g*s_anx(0), g*s_dep(0), s_anx*s_dep(0) # all factore uncorrelated

  CONSTRAIN = (1, 7, a2), (4, 10, a3) # # essential tau-equivalence: forces equal loadings to stabilize 2-item facets
'

bi_anx_dep_2_fit <- mirt(data = neuro_items, model = bi_anx_dep_2_spec, itemtype = "graded", SE = TRUE, verbose = FALSE)

---
### Comparing the Models
##### Model Summaries

In [6]:
%%R
print(summary(grm_fit))

              g    h2
NEORAW_01 0.490 0.240
NEORAW_06 0.586 0.343
NEORAW_11 0.706 0.498
NEORAW_16 0.687 0.472
NEORAW_21 0.708 0.501
NEORAW_26 0.794 0.630
NEORAW_31 0.671 0.451
NEORAW_36 0.521 0.271
NEORAW_41 0.748 0.560
NEORAW_46 0.579 0.335
NEORAW_51 0.662 0.439
NEORAW_56 0.617 0.381

           SE.g
NEORAW_01 0.027
NEORAW_06 0.025
NEORAW_11 0.020
NEORAW_16 0.021
NEORAW_21 0.021
NEORAW_26 0.017
NEORAW_31 0.021
NEORAW_36 0.027
NEORAW_41 0.019
NEORAW_46 0.025
NEORAW_51 0.024
NEORAW_56 0.024

SS loadings:  5.122 
Proportion Var:  0.427 

Factor correlations: 

  g
g 1
$rotF
                  g
NEORAW_01 0.4900426
NEORAW_06 0.5858703
NEORAW_11 0.7060346
NEORAW_16 0.6871869
NEORAW_21 0.7078688
NEORAW_26 0.7936279
NEORAW_31 0.6711985
NEORAW_36 0.5209752
NEORAW_41 0.7484748
NEORAW_46 0.5791613
NEORAW_51 0.6622647
NEORAW_56 0.6171602

$SE.F
                SE.g
NEORAW_01 0.02734562
NEORAW_06 0.02476342
NEORAW_11 0.02011941
NEORAW_16 0.02086693
NEORAW_21 0.02073130
NEORAW_26 0.01728942
NEORAW_

In [7]:
%%R
print(summary(bi_anx_dep_all_fit))

              g s_anx   s_dep    h2
NEORAW_01 0.450 0.409         0.370
NEORAW_06 0.589               0.347
NEORAW_11 0.709               0.503
NEORAW_16 0.634        0.4982 0.651
NEORAW_21 0.685 0.208         0.513
NEORAW_26 0.789        0.1093 0.634
NEORAW_31 0.618 0.490         0.622
NEORAW_36 0.525               0.276
NEORAW_41 0.765       -0.0365 0.587
NEORAW_46 0.534        0.3810 0.430
NEORAW_51 0.682               0.466
NEORAW_56 0.625               0.390

           SE.g SE.s_anx SE.s_dep
NEORAW_01 0.029    0.054         
NEORAW_06 0.025                  
NEORAW_11 0.020                  
NEORAW_16 0.031             0.070
NEORAW_21 0.022    0.046         
NEORAW_26 0.018             0.042
NEORAW_31 0.030    0.062         
NEORAW_36 0.027                  
NEORAW_41 0.020             0.046
NEORAW_46 0.027             0.047
NEORAW_51 0.024                  
NEORAW_56 0.024                  

SS loadings:  4.931 0.45 0.407 
Proportion Var:  0.411 0.038 0.034 

Factor correlations

In [8]:
%%R 
print(summary(bi_anx_dep_2_fit))

              g s_anx s_dep    h2
NEORAW_01 0.459 0.465       0.427
NEORAW_06 0.586             0.344
NEORAW_11 0.705             0.497
NEORAW_16 0.653       0.394 0.582
NEORAW_21 0.702             0.493
NEORAW_26 0.794             0.630
NEORAW_31 0.639 0.403       0.571
NEORAW_36 0.520             0.271
NEORAW_41 0.752             0.565
NEORAW_46 0.540       0.438 0.483
NEORAW_51 0.671             0.450
NEORAW_56 0.620             0.385

           SE.g SE.s_anx SE.s_dep
NEORAW_01 0.028    0.033         
NEORAW_06 0.025                  
NEORAW_11 0.020                  
NEORAW_16 0.022             0.034
NEORAW_21 0.021                  
NEORAW_26 0.017                  
NEORAW_31 0.021    0.033         
NEORAW_36 0.027                  
NEORAW_41 0.019                  
NEORAW_46 0.031             0.030
NEORAW_51 0.024                  
NEORAW_56 0.024                  

SS loadings:  4.973 0.379 0.346 
Proportion Var:  0.414 0.032 0.029 

Factor correlations: 

      g s_anx s_dep
g

In [9]:
%%R
print(summary(bi_4fac_all_fit))

              g s_anx s_dep s_selfcon s_vul    h2
NEORAW_01 0.339 0.466                       0.332
NEORAW_06 0.492                 0.116       0.255
NEORAW_11 0.609                       0.439 0.564
NEORAW_16 0.500       0.610                 0.622
NEORAW_21 0.591 0.331                       0.458
NEORAW_26 0.663       0.237                 0.496
NEORAW_31 0.494 0.537                       0.532
NEORAW_36 0.460                             0.212
NEORAW_41 0.639       0.119                 0.423
NEORAW_46 0.422       0.455                 0.385
NEORAW_51 0.581                       0.187 0.372
NEORAW_56 0.508                 0.195       0.296

           SE.g SE.s_anx SE.s_dep SE.s_selfcon SE.s_vul
NEORAW_01 0.028    0.045                               
NEORAW_06 0.026                           0.22         
NEORAW_11 0.045                                   0.135
NEORAW_16 0.028             0.052                      
NEORAW_21 0.022    0.050                               
NEORAW_26 0.0

##### M2 Statistics

In [10]:
%%R
M2(grm_fit, type = "C2")

            M2 df p      RMSEA    RMSEA_5   RMSEA_95      SRMSR       TLI
stats 426.3428 54 0 0.07589751 0.06925802 0.08264562 0.06138378 0.9443336
            CFI
stats 0.9544548


In [11]:
%%R
M2(bi_anx_dep_all_fit, type="C2")

            M2 df p      RMSEA   RMSEA_5   RMSEA_95      SRMSR      TLI
stats 231.6951 47 0 0.05729697 0.0500316 0.06473868 0.04978421 0.968275
           CFI
stats 0.977408


In [12]:
%%R
M2(bi_anx_dep_2_fit, type = "C2")

            M2 df p      RMSEA    RMSEA_5   RMSEA_95     SRMSR       TLI
stats 270.9399 52 0 0.05930806 0.05241832 0.06634912 0.0508785 0.9660089
            CFI
stats 0.9732191


In [13]:
%%R
M2(bi_4fac_all_fit, type="C2")

            M2 df p      RMSEA    RMSEA_5   RMSEA_95     SRMSR     TLI
stats 185.1588 43 0 0.05255397 0.04488703 0.06042417 0.1066879 0.97331
            CFI
stats 0.9826111


##### Anova

In [14]:
%%R
anova(grm_fit, bi_anx_dep_2_fit, bi_anx_dep_all_fit, bi_4fac_all_fit)

                        AIC    SABIC       HQ      BIC    logLik       X2 df
grm_fit            34367.65 34482.37 34482.66 34672.95 -17123.82            
bi_anx_dep_2_fit   34289.41 34407.96 34408.26 34604.89 -17082.71   82.237  2
bi_anx_dep_all_fit 34272.96 34401.07 34401.39 34613.89 -17069.48   26.449  5
bi_4fac_all_fit    34623.75 34759.51 34759.86 34985.03 -17240.88 -342.793  4
                     p
grm_fit               
bi_anx_dep_2_fit     0
bi_anx_dep_all_fit   0
bi_4fac_all_fit    NaN


---
### Model Selection Rationale:

The `bi_anx_dep_2_fit` model was selected as the optimal latent structure based on the following evaluation:

- **Exclusion of Overparameterized Models**: The 4-facet bifactor model `bi_4fac_all_fit` was discarded as it was overparameterized (see ANOVA results) and contained multiple near-zero loadings (see model summary). Similarly, the full facet model containing only anxienty and depression `bi_anx_dep_all_fit` was rejected due to unstable loadings approaching zero (see model summary).

- **Superiority of Fit and Parsimony**: Among the remaining simpler models, the bifactor model containing only two items per facet (`bi_anx_dep_2_fit`) demonstrated superior fit compared to the baseline unidimensional model (`grm_fit`), as evidenced by significantly lower AIC and BIC indices and a significant likelihood ratio test (see ANOVA). 

Consequently, this model was retained in preference to more complex alternatives due to its superior structural parsimony, and in preference to the simpler unidimensional model due to its superior fit and satisfactory specific factor loadings.
 
